<a href="https://colab.research.google.com/github/wmjx691/rental-market-analyzer/blob/main/scraper_withGeo_Fencing_asset_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 環境建置與依賴套件 (Environment Setup)
配置爬蟲運行環境，包含 Selenium WebDriver、無頭瀏覽器 (Headless Browser) 設定，以及開源地理編碼套件 `geopy` 的安裝，確保後續網頁渲染與距離計算順利進行。

In [ ]:
# @title 1. 安裝必要套件、瀏覽器驅動與中文字型
# 安裝 selenium 和 google drive 相關套件
!pip install selenium gspread oauth2client webdriver_manager

# --- 新增：安裝地理資訊處理套件 ---
!pip install geopy

# 1. 安裝 Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y

# 2. 關鍵修正：安裝中文字型 (解決截圖方塊字問題)
!apt-get install -y fonts-noto-cjk

print("✅ 環境安裝完成！Geopy 已就緒。")

## 2. Google 雲端服務授權 (GCP Authentication)
建立與 Google Sheets 的安全連線 (OAuth 2.0)。此步驟負責打通資料庫，讓最終清洗完畢的黃金資料能無縫回寫至雲端試算表，實現資料追蹤與自動化更新。

*執行時會跳出視窗要求權限，請點選「允許」。*

In [ ]:
# @title 2. Google 帳號授權與試算表連線
from google.colab import auth
import gspread
from google.auth import default

# 進行身分驗證
auth.authenticate_user()       # 這一行會跳出彈窗要你登入
creds, _ = default()           # 這是暫時性的 Session 憑證
gc = gspread.authorize(creds)

print("Google 帳號授權成功！準備開始爬蟲...")

## 3. 系統參數配置與核心函式 (Config & Core Functions)
定義全域變數以提升程式碼的可維護性與擴展性（Config-Driven）。
使用者可在此自定義**目標行政區**、**物件型態**、**抓取數量上限**以及**距離參考點 (Geo-Fencing 上限)**。同時初始化 Nominatim 地理編碼函式，處理基礎的地址模糊化解析。

In [ ]:
# @title 3. 初始化：全域設定、函式定義與載入資料庫 (Run Once)
import time
import pandas as pd
import re
import pytz
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from IPython.display import Image, display
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

# ==========================================
# ⚙️ 全域設定區 (CONFIG) - 修改這裡即可改變爬蟲目標
# ==========================================

# 1. 檔案名稱設定 (建議換新名稱以區隔舊格式)
SHEET_NAME_NEW = 'TARGET_SHEET_NAME'
SHEET_NAME_OLD = 'TARGET_SHEET_NAME' # 用於備份或遷移

# 2. 地理位置與目標設定
TARGET_CITY = "台北市"                              # 目標縣市 (用於地址解析與補全)
TARGET_REGION_CODE = "1"                          # 地區代碼 (1=台北, 3=新北, 17=高雄, 15=台南...)
TARGET_DISTRICTS = ["中正區", "中山區", "大同區"]    # 目標行政區列表 (可無限新增)
RENTAL_TYPES = ["整層住家", "獨立套房", "分租套房"]  # 選項：整層住家, 獨立套房, 分租套房, 雅房

# 3. 抓取數量上限設定
MAX_ITEMS_PER_TYPE = 200 # 輸入整數，或輸入 'MAX' 抓取全部

# 4. 距離限制設定 (單位：公里)
# 預設大於此距離的物件將被排除。
# 若距離無法計算 (N/A)，則無條件保留供人工確認。
# 若要取消距離限制，請填寫 'MAX'
MAX_DISTANCE_KM = 2.0

# 5. 參考點設定 (Anchor)
ANCHOR_NAME = "台北車站"
ANCHOR_COORDS = (25.047772,121.516867)

# 6. 其他系統設定
TW_TZ = pytz.timezone('Asia/Taipei')

# ==========================================

# --- 1. 設定瀏覽器選項 ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# --- 2. 工具函式定義 ---
def click_element_by_text(driver, text):
    try:
        xpath = f"//label[contains(text(),'{text}')] | //span[contains(text(),'{text}')] | //li[contains(text(),'{text}')]"
        element = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)
        return True
    except: return False

def load_data_from_sheet(sheet_name):
    print(f"📂 正在讀取工作表: {sheet_name} ...")
    try:
        sh = gc.open(sheet_name)
        worksheet = sh.sheet1
        data = worksheet.get_all_records()
        if not data: return {}

        df = pd.DataFrame(data)
        if '物件ID' not in df.columns: return {}

        data_map = {}
        for index, row in df.iterrows():
            pid = str(row['物件ID'])
            data_map[pid] = row.to_dict()
        print(f"✅ 成功載入 {len(data_map)} 筆紀錄。")
        return data_map
    except Exception as e:
        print(f"   -> 讀取失敗或是新檔案: {e}")
        return {}

def save_full_data(df, sheet_name):
    print(f"💾 正在儲存至: {sheet_name} ...")
    try:
        try: sh = gc.open(sheet_name)
        except: sh = gc.create(sheet_name)

        worksheet = sh.sheet1
        worksheet.clear()
        worksheet.append_row(df.columns.tolist())
        worksheet.append_rows(df.values.tolist())
        print(f"✅ 儲存完成！共 {len(df)} 筆。連結: {sh.url}")
    except Exception as e:
        print(f"❌ 儲存失敗: {e}")

# --- 3. 強效地理計算函式 ---
def get_distance_from_anchor(address_str):
    """
    輸入地址，回傳 (距離, 緯度, 經度)。
    使用全域變數 TARGET_CITY 來補全地址。
    """
    geolocator = Nominatim(user_agent="my_rental_scraper_custom_v5")
    # 使用設定檔中的城市名稱
    if TARGET_CITY not in address_str:
        full_addr = TARGET_CITY + address_str
    else:
        full_addr = address_str

    try:
        # 第一次嘗試：完整地址
        location = geolocator.geocode(full_addr, timeout=5)
        # 如果失敗 (None)，嘗試模糊化
        if not location:
            # 移除數字、號、樓、之，保留路名
            fuzzy_addr = re.sub(r'\d+[號樓之F].*', '', full_addr)
            if len(fuzzy_addr) > len(TARGET_CITY) + 1:
                location = geolocator.geocode(fuzzy_addr, timeout=5)

        if location:
            target_point = (location.latitude, location.longitude)
            distance = geodesic(ANCHOR_COORDS, target_point).km
            return round(distance, 2), location.latitude, location.longitude
        else:
            return "N/A", "N/A", "N/A"
    except Exception as e:
        return "N/A", "N/A", "N/A"

# 預設載入資料庫
history_database = load_data_from_sheet(SHEET_NAME_NEW)

## 4-1. 爬蟲階段一：自動化網頁擷取 (Web Scraping Engine)
針對目標租屋網實作的反爬蟲突破機制：
* **多類別輪詢**：自動切換整層住家、獨立/分租套房。
* **無限滾動突破**：利用 DOM 操作與鍵盤模擬 (`Keys.END`) 強制載入動態渲染的物件。
* **唯一 ID 記憶池**：防止「假下一頁按鈕」造成的死迴圈，精準萃取原始網頁文本至記憶體緩存中。

In [ ]:
# @title 4-1. 執行爬蟲：第一階段 (目標租屋網抓取)
if 'history_database' not in globals():
    print("⚠️ 請先執行 Cell 3！")
else:
    limit_text = "無上限" if MAX_ITEMS_PER_TYPE == 'MAX' else f"{MAX_ITEMS_PER_TYPE} 筆"
    dist_text = "無上限" if MAX_DISTANCE_KM == 'MAX' else f"{MAX_DISTANCE_KM} km"
    print(f"🚀 啟動爬蟲 | 目標: {TARGET_CITY} {TARGET_DISTRICTS} | 數量上限: {limit_text} | 距離上限: {dist_text}")

    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.common.action_chains import ActionChains

    chrome_options_optimized = Options()
    for arg in [
        '--headless', '--no-sandbox', '--disable-dev-shm-usage',
        '--window-size=1920,1080', '--disable-blink-features=AutomationControlled'
    ]:
        chrome_options_optimized.add_argument(arg)

    prefs = {"profile.managed_default_content_settings.images": 2, "profile.default_content_setting_values.notifications": 2}
    chrome_options_optimized.add_experimental_option("prefs", prefs)
    chrome_options_optimized.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options_optimized)

    driver.set_page_load_timeout(20)
    raw_data_list = []

    # --- 第一階段：掃描迴圈 ---
    for current_rental_type in RENTAL_TYPES:
        print(f"\n🏎️ [階段一] 正在掃描: {current_rental_type} ...")

        target_url = f"https://rental.example.com.tw/?region={TARGET_REGION_CODE}"

        try:
            driver.get(target_url)
        except Exception:
            print("   ⚠️ 網頁載入超時 (20s)，強制中斷載入並嘗試繼續操作...")
            try: driver.execute_script("window.stop();")
            except: pass

        try:
            WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.close, i.close, .TIGerm"))).click()
        except: pass
        time.sleep(1)

        for district in TARGET_DISTRICTS:
            click_element_by_text(driver, district)

        if not click_element_by_text(driver, current_rental_type):
            print(f"   ⚠️ 無法選取 '{current_rental_type}'，跳過。")
            continue

        try:
            search_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'搜尋')] | //div[contains(@class,'search')]//button")))
            driver.execute_script("arguments[0].click();", search_btn)
        except: pass

        print("   ⏳ 等待列表載入...")
        time.sleep(3)

        page_num = 1
        items_in_this_type = 0
        unique_ids_this_type = set()

        while True:
            # 滾動確保網頁載入
            print(f"      -> 第 {page_num} 頁: 正在向下捲動...", end="\r")
            try:
                body = driver.find_element(By.TAG_NAME, "body")
                for i in range(10):
                    body.send_keys(Keys.END)
                    time.sleep(1.0)
                    next_btns = driver.find_elements(By.XPATH, "//a[contains(@class, 'pageNext')]")
                    if next_btns and next_btns[0].is_displayed():
                        break
            except Exception: pass

            items = driver.find_elements(By.CSS_SELECTOR, ".vue-list-rent-item, .listing-recommend-item, div[class*='item']")

            current_page_new_items = 0
            for item in items:
                try:
                    if item.size['height'] < 80: continue
                    link = "N/A"
                    try: link = item.find_element(By.TAG_NAME, "a").get_attribute("href")
                    except: pass

                    post_id = "N/A"
                    id_match = re.search(r'(\d{7,8})', str(link))
                    if id_match: post_id = id_match.group(1)
                    if post_id == "N/A" or post_id in unique_ids_this_type: continue

                    unique_ids_this_type.add(post_id)

                    raw_data_list.append({
                        "物件ID": post_id,
                        "連結": link,
                        "原始文字": item.text,
                        "物件型態": current_rental_type
                    })
                    current_page_new_items += 1
                except: continue

            items_in_this_type += current_page_new_items
            print(f"      -> 第 {page_num} 頁掃描完畢: 本頁新增 {current_page_new_items} 筆 (累積 {items_in_this_type} 筆)           ", end="\r")

            if MAX_ITEMS_PER_TYPE != 'MAX' and isinstance(MAX_ITEMS_PER_TYPE, int):
                if items_in_this_type >= MAX_ITEMS_PER_TYPE:
                    print(f"\n      🛑 已達到設定的抓取上限 ({MAX_ITEMS_PER_TYPE} 筆)，停止翻頁。")
                    break

            if current_page_new_items == 0:
                print(f"\n      🛑 本頁未發現新物件，停止翻頁。")
                break

            try:
                next_btns = driver.find_elements(By.XPATH, "//a[contains(@class, 'pageNext') or contains(text(), '下一頁')]")
                if not next_btns: break

                next_btn = next_btns[0]
                btn_class = next_btn.get_attribute("class") or ""
                if "disabled" in btn_class or "last" in btn_class: break

                driver.execute_script("arguments[0].click();", next_btn)
                page_num += 1
                time.sleep(3)

            except Exception as e: break

        print(f"\n   ✅ {current_rental_type} 掃描完成。")

    driver.quit()

## 4-2. 爬蟲階段二：資料清洗與地理圍欄 (Data Pipeline & Geo-Fencing)
讀取階段一的記憶體快取，進行二次處理：
* **正規化萃取**：利用正則表達式 (Regex) 精準抓取價格、坪數並進行極端值過濾。
* **狀態即時監控**：導入 `clear_output` 實現單點即時進度刷新，視覺化監控 API 請求狀態。
* **開源地理運算**：使用 Nominatim API 呼叫地址經緯度，並以 Haversine 公式計算物件與參考點的距離，剔除超標物件後，合併歷史資料匯出。

In [ ]:
# @title 4-2. 執行爬蟲：第二階段 (資料處理與地理距離計算) - 🔄 單筆刷新版
from IPython.display import clear_output
import time
from datetime import datetime
import pandas as pd
import re

if 'raw_data_list' not in globals() or not raw_data_list:
    print("⚠️ 找不到 raw_data_list，請先執行 4-1 完成第一階段掃描！")
else:
    processed_data_list = []
    today_str = datetime.now(TW_TZ).strftime("%Y-%m-%d")
    last_check_time = datetime.now(TW_TZ).strftime("%Y-%m-%d %H:%M:%S")
    district_regex_str = f"({'|'.join(TARGET_DISTRICTS)})"

    unique_raw_data = {item['物件ID']: item for item in raw_data_list}.values()
    total_items = len(unique_raw_data)
    processed_count = 0
    filtered_by_distance_count = 0

    for raw in unique_raw_data:
        processed_count += 1

        try:
            post_id = raw['物件ID']
            full_text = raw['原始文字']
            link = raw['連結']
            item_type = raw['物件型態']

            # --- 🖥️ 畫面刷新與狀態標頭 ---
            clear_output(wait=True)
            print(f"⚙️ [階段二] 開始處理 {len(raw_data_list)} 筆原始資料 (解析 + 算距離)...\n")
            print(f"進度: {processed_count}/{total_items} | 物件ID: {post_id}")
            text_snippet = full_text.replace('\n', ' ')[:40]
            print(f"   -> 原始文字片段: {text_snippet}...")

            if len(full_text) < 10:
                print("   -> ⚠️ 文字過短，跳過。")
                time.sleep(0.3)
                continue

            price = "N/A"
            match = re.search(r'(\d{1,3}(,\d{3})*)\s*元/月', full_text)
            if match: price = match.group(0).replace("元/月", "").strip()
            else:
                print("   -> ⚠️ 找不到價格，跳過。")
                time.sleep(0.3)
                continue

            area = "N/A"
            match = re.search(r'(\d+\.?\d*)\s*坪', full_text)
            if match: area = match.group(1)

            try:
                if area != "N/A":
                    area_f = float(area)
                    if item_type == "整層住家" and area_f < 15: continue
                    if "套房" in item_type and area_f < 4: continue
            except: pass

            lines = full_text.split('\n')
            title = lines[0] if lines else "N/A"
            if len(title) < 5 and len(lines) > 1: title = lines[1]

            site_update_text = "N/A"
            for line in lines:
                if any(k in line for k in ["更新", "發佈", "前", "昨天", "剛"]):
                    if len(line) < 15: site_update_text = line; break

            location_full, city, district, address_road = "N/A", TARGET_CITY, "N/A", "N/A"
            for line in lines:
                if "區" in line and ("市" in line or "-" in line):
                    location_full = line; break
            if location_full != "N/A":
                parts = re.split(r'[-/ ]', location_full)
                for p in parts:
                    if "區" in p: district = p
                    if "路" in p or "街" in p: address_road = p
            if district == "N/A":
                m = re.search(district_regex_str, full_text)
                if m: district = m.group(1)

            dist_km, lat, lon = "N/A", "N/A", "N/A"
            is_processed = False

            if post_id in history_database:
                old = history_database[post_id]
                if '距離(km)' in old and old['距離(km)'] != "N/A":
                    dist_km = old['距離(km)']
                    lat = old.get('緯度', "N/A")
                    lon = old.get('經度', "N/A")
                    is_processed = True
                    print(f"   -> ♻️ 歷史資料已存在，直接沿用距離: {dist_km} km")

            if not is_processed and address_road != "N/A":
                search_target = location_full if location_full != "N/A" else f"{district}{address_road}"
                search_target = search_target.replace("-", "").replace("/", "")

                print(f"   -> 📡 準備呼叫 API 解析地址: {search_target} ...", end=" ")
                dist_km, lat, lon = get_distance_from_anchor(search_target)
                print(f"✅ 完成！距離: {dist_km} km")

                # 嚴格遵守 OSM 的 1.5 秒限制，同時讓使用者看清楚執行結果
                time.sleep(1.5)

            # --- 距離過濾邏輯 ---
            if MAX_DISTANCE_KM != 'MAX' and isinstance(MAX_DISTANCE_KM, (int, float)):
                if dist_km != "N/A":
                    try:
                        if float(dist_km) > MAX_DISTANCE_KM:
                            filtered_by_distance_count += 1
                            print(f"   -> 🛑 距離 {dist_km} km 超過上限 {MAX_DISTANCE_KM} km，已排除。")
                            time.sleep(0.5) # 稍微暫停讓肉眼看到排除訊息
                            continue
                    except: pass

            # --- 紀錄存活的天數 ---
            first_seen, days_mkt = today_str, 0
            if post_id in history_database:
                old = history_database[post_id]
                if '首次發現日' in old and old['首次發現日']:
                    first_seen = old['首次發現日']
                    try: days_mkt = (datetime.strptime(today_str, "%Y-%m-%d") - datetime.strptime(first_seen, "%Y-%m-%d")).days
                    except: pass

            processed_data_list.append({
                "物件ID": post_id,
                "最後更新時間": last_check_time,
                "首次發現日": first_seen,
                "上架已持續天數": days_mkt,
                "網站顯示更新": site_update_text,
                "物件型態": item_type,
                "標題": title,
                "價格": price,
                "坪數": area,
                "距離(km)": dist_km,
                "緯度": lat,
                "經度": lon,
                "完整顯示地址": location_full,
                "路段/地址": address_road,
                "連結": link
            })
            print("   -> 📥 成功加入處理清單。")

            # 若不是呼叫 API (歷史沿用)，稍微停頓一下讓畫面不會閃爍過快
            if is_processed: time.sleep(0.1)

        except Exception as e:
            print(f"   -> ❌ 發生未預期錯誤: {e}")
            time.sleep(1)
            continue

    # --- 迴圈結束，顯示最終統計結果 ---
    clear_output(wait=True)
    print(f"\n📊 統計: 處理完成，共產出 {len(processed_data_list)} 筆有效資料 (因距離過遠排除 {filtered_by_distance_count} 筆)，開始存檔...")
    df_new = pd.DataFrame(processed_data_list)
    if not df_new.empty:
        df_new['廣告投放數'] = df_new.groupby(['價格', '坪數'])['物件ID'].transform('count')

        final_map = history_database.copy()
        for idx, row in df_new.iterrows():
            final_map[row['物件ID']] = row.to_dict()

        df_final = pd.DataFrame(list(final_map.values()))
        if '最後更新時間' in df_final.columns:
            df_final = df_final.sort_values(by='最後更新時間', ascending=False)

        cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "物件型態", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]
        df_final = df_final[[c for c in cols if c in df_final.columns]]

        save_full_data(df_final, SHEET_NAME_NEW)
        history_database = final_map
    else:
         print("⚠️ 過濾後沒有剩餘資料可供存檔。")

---

### 如何執行你的「一週測試計畫」

既然你要測試一週，且每三天執行一次，操作流程如下：

1. **第一次（今天）：**
* 登入你的 Google Drive，建立一個 Colab 筆記本。
* 將上述三段代碼貼入。
* 依序點擊「播放鍵」執行 Cell 1, 2, 3, 4-1, 4-2。
* 執行完後，去你的 Google Drive 根目錄找找看，會有一個你在CONFIG設定中名為 **`SHEET_NAME_NEW`** 的試算表。打開來確認資料是否正確。


2. **第二次（三天後）：**
* 打開這個 Colab 網頁。
* **重要：** 因為 Colab 會重置環境，所以你必須**再次點擊 Cell 1, 2, 3, 4-1, 4-2**。
* 程式會自動把新的資料「新增」到那張試算表的下面，不會覆蓋舊資料。


3. **第三次（六天後）：**
* 重複上述動作。



### 提醒（關於目標租屋網的反爬蟲）

在 Colab 的「無頭模式（Headless）」下，瀏覽器特徵非常明顯，租屋網這種網站有時會直接阻擋（你可能會看到程式跑完但說「抓到 0 筆物件」）。

* **如果發生這種情況**：代表目標租屋網擋掉了 Colab 的 IP 或特徵。這時候最簡單的解法，還是回到我一開始提供的 **PC 本地端執行**（因為你在本地有視窗介面，比較像真人）。